In [10]:
!pip install -q pandas transformers google-genai

In [11]:
import queue
import threading
import time
from transformers import pipeline

# Load AI Models
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

intent_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

customer_message = "I am very much disappointed with your service."

result = sentiment_analyzer(customer_message)
print(result)

labels = [
    "complaint",
    "purchase",
    "query"
]

intent_result = intent_classifier(
    "My internet is not working.",
    candidate_labels=labels
)

print(intent_result)

# Define Coaching Logic
def generate_coaching_feedback(agent_message, customer_message):
    feedback = []

    # Sentiment check
    sentiment = sentiment_analyzer(customer_message)[0]

    if sentiment["label"] == "NEGATIVE":
        feedback.append(
            "Customer seems upset. Use empathy and reassure them."
        )

    # Intent detection
    intents = [
        "complaint",
        "query",
        "purchase",
        "technical issue",
        "feedback"
    ]

    intent_result = intent_classifier(
        customer_message,
        candidate_labels=intents
    )

    top_intent = intent_result["labels"][0]

    if top_intent == "complaint":
        feedback.append(
            "Acknowledge the issue clearly and offer a resolution path."
        )
    elif top_intent == "technical issue":
        feedback.append(
            "Guide step-by-step troubleshooting, avoid jargon."
        )
    elif top_intent == "purchase":
        feedback.append(
            "Highlight product benefits and reassure about value."
        )
    elif top_intent == "query":
        feedback.append(
            "Answer concisely and check if customer needs more details."
        )
    elif top_intent == "feedback":
        feedback.append(
            "Thank the customer and note their input."
        )

    # Agent coaching
    if "sorry" not in agent_message.lower() and sentiment["label"] == "NEGATIVE":
        feedback.append(
            "Consider apologizing to show empathy."
        )

    return feedback


# Call the function
agent_message = "I understand your concern. Let me check the issue."

feedback = generate_coaching_feedback(agent_message, customer_message)

print("\nCoaching Feedback:")
for item in feedback:
    print("-", item)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997332692146301}]
{'sequence': 'My internet is not working.', 'labels': ['complaint', 'query', 'purchase'], 'scores': [0.773064136505127, 0.17086835205554962, 0.05606752634048462]}

Coaching Feedback:
- Customer seems upset. Use empathy and reassure them.
- Acknowledge the issue clearly and offer a resolution path.
- Consider apologizing to show empathy.


In [12]:
from dataclasses import dataclass, field
from typing import List


# 1. DATA MODELS

@dataclass
class Message:
    """Represents one message in the conversation."""
    speaker: str
    text: str


@dataclass
class ConversationState:
    """Stores the complete conversation and current customer risk."""
    history: List[Message] = field(default_factory=list)
    sentiment: str = "unknown"
    urgency: str = "unknown"
    escalation_risk: str = "unknown"
    key_issue: str = ""

    def add_message(self, speaker: str, text: str):
        """Add a new message to the conversation history."""
        self.history.append(
            Message(
                speaker=speaker,
                text=text
            )
        )


@dataclass
class CoachingFeedback:
    """Stores feedback about the agent's response."""
    tone_score: int
    empathy_score: int
    clarity_score: int
    coaching_tip: str

In [13]:
import os
import json
import time

from getpass import getpass
from typing import List, Optional

from google import genai


os.environ["GEMINI_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)

Enter your Gemini API key: ··········


In [17]:
# 2. AI COACH

class AICoach:

    def __init__(self):
        """
        Create the Gemini client.
        """

        api_key = os.getenv("GEMINI_API_KEY")

        if not api_key:
            raise ValueError(
                "GEMINI_API_KEY environment variable is not set."
            )

        self.client = genai.Client(
            api_key=api_key
        )

        # Gemini model
        self.model = "gemini-3.6-flash"

    # ==========================================
    # HELPER FUNCTION
    # ==========================================

    def _parse_json(self, text: str) -> dict:

        text = text.strip()

        # Remove markdown code fences if Gemini adds them
        if text.startswith("```"):
            text = text.replace("```json", "")
            text = text.replace("```", "")
            text = text.strip()

        try:

            result = json.loads(text)

        except json.JSONDecodeError as e:

            print("\nCould not parse Gemini response as JSON.")

            print("Raw response:")
            print(text)

            raise ValueError(
                f"Invalid JSON returned by Gemini: {e}"
            )

        return result

    # ==========================================
    # GEMINI REQUEST WITH ERROR HANDLING
    # ==========================================

    def _generate_content(self, prompt: str):

        for attempt in range(3):

            try:

                response = self.client.models.generate_content(
                    model=self.model,
                    contents=prompt
                )

                return response

            except Exception as e:

                error_message = str(e)

                if (
                    "503" in error_message
                    or "UNAVAILABLE" in error_message
                ):

                    if attempt < 2:

                        print(
                            f"\nGemini server is temporarily busy."
                            f" Retrying... ({attempt + 1}/3)"
                        )

                        time.sleep(3)

                    else:

                        raise RuntimeError(
                            "Gemini is temporarily unavailable. "
                            "Please try again after a few minutes."
                        )

                else:

                    raise e

    # ==========================================
    # 1. ANALYZE CUSTOMER MESSAGE
    # ==========================================

    def analyze_customer_message(
        self,
        customer_message: str
    ) -> dict:

        prompt = f"""
You are an AI customer-support risk analyzer.

Analyze the following customer message.

Customer message:
{customer_message}

Return ONLY valid JSON.

Use exactly this structure:

{{
    "sentiment": "positive|neutral|negative",
    "urgency": "low|medium|high",
    "escalation_risk": "low|medium|high",
    "key_issue": "short description of the main issue"
}}

Do not add explanations outside the JSON.
"""

        response = self._generate_content(
            prompt
        )

        text = response.text

        return self._parse_json(text)

    # ==========================================
    # 2. EVALUATE AGENT RESPONSE
    # ==========================================

    def evaluate_agent_response(
        self,
        customer_message: str,
        agent_message: str
    ) -> CoachingFeedback:

        prompt = f"""
You are an AI customer-support coach.

Evaluate the agent's response to the customer.

Customer message:
{customer_message}

Agent response:
{agent_message}

Score the agent from 1 to 10 for:

1. Tone
2. Empathy
3. Clarity

Then provide ONE concrete coaching tip.

Return ONLY valid JSON using exactly this structure:

{{
    "tone_score": 1,
    "empathy_score": 1,
    "clarity_score": 1,
    "coaching_tip": "one concrete coaching tip"
}}

Do not add any explanation outside the JSON.
"""

        response = self._generate_content(
            prompt
        )

        text = response.text

        result = self._parse_json(
            text
        )

        return CoachingFeedback(
            tone_score=int(
                result["tone_score"]
            ),

            empathy_score=int(
                result["empathy_score"]
            ),

            clarity_score=int(
                result["clarity_score"]
            ),

            coaching_tip=result[
                "coaching_tip"
            ]
        )

    # ==========================================
    # 3. SUGGEST REPLY
    # ==========================================

    def suggest_reply(
        self,
        customer_message: str,
        conversation_history: Optional[List[Message]] = None
    ) -> str:

        history_text = ""

        if conversation_history:

            history_text = "\n".join(
                f"{message.speaker}: {message.text}"
                for message in conversation_history
            )

        prompt = f"""
You are an expert customer-support agent.

Create a professional, empathetic and concise reply to the customer.

Customer message:
{customer_message}

Previous conversation:
{history_text}

The customer may be frustrated.

Requirements:

- Acknowledge the customer's concern.
- Show empathy.
- Clearly explain the next step if possible.
- Do not make unsupported promises.
- Keep the response concise.
- Return only the reply text.
"""

        response = self._generate_content(
            prompt
        )

        return response.text.strip()

In [18]:
# 3. REAL-TIME COACHING SESSION

class RealTimeCoachingSession:

    def __init__(self):

        self.state = ConversationState()

        self.coach = AICoach()

        # Store the previous customer message
        # so the next agent response can be evaluated
        self.last_customer_message = ""

    # ==========================================
    # CUSTOMER MESSAGE
    # ==========================================

    def on_customer_message(self, message: str):

        # Add customer message to conversation
        self.state.add_message(
            "customer",
            message
        )

        # Remember latest customer message
        self.last_customer_message = message

        # Analyze customer message
        analysis = self.coach.analyze_customer_message(
            message
        )

        # Update conversation state
        self.state.sentiment = analysis["sentiment"]
        self.state.urgency = analysis["urgency"]
        self.state.escalation_risk = analysis["escalation_risk"]
        self.state.key_issue = analysis["key_issue"]

        # ======================================
        # If escalation risk is HIGH
        # generate suggested reply
        # ======================================

        suggested_reply = None

        if self.state.escalation_risk.lower() == "high":

            suggested_reply = self.coach.suggest_reply(
                customer_message=message,
                conversation_history=self.state.history
            )

        # Return everything to frontend
        return {
            "sentiment": self.state.sentiment,
            "urgency": self.state.urgency,
            "escalation_risk": self.state.escalation_risk,
            "key_issue": self.state.key_issue,
            "suggested_reply": suggested_reply
        }

    # ==========================================
    # AGENT MESSAGE
    # ==========================================

    def on_agent_message(self, message: str):

        # Add agent message to conversation
        self.state.add_message(
            "agent",
            message
        )

        # Evaluate agent response
        feedback = self.coach.evaluate_agent_response(
            customer_message=self.last_customer_message,
            agent_message=message
        )

        # Return feedback to frontend
        return {
            "tone_score": feedback.tone_score,
            "empathy_score": feedback.empathy_score,
            "clarity_score": feedback.clarity_score,
            "coaching_tip": feedback.coaching_tip
        }

In [19]:
# 4. TEST MODE

session = RealTimeCoachingSession()

customer_result = session.on_customer_message(
    "I have been waiting for hours and my internet still isn't working!"
)

print("\n--- Customer Analysis ---")

print(
    "Sentiment:",
    customer_result["sentiment"]
)

print(
    "Urgency:",
    customer_result["urgency"]
)

print(
    "Escalation Risk:",
    customer_result["escalation_risk"]
)

print(
    "Key Issue:",
    customer_result["key_issue"]
)

if customer_result["suggested_reply"]:

    print("\n--- Suggested Reply ---")

    print(
        customer_result["suggested_reply"]
    )


agent_result = session.on_agent_message(
    "Please restart your router and check again."
)

print("\n--- Agent Coaching ---")

print(
    "Tone Score:",
    agent_result["tone_score"]
)

print(
    "Empathy Score:",
    agent_result["empathy_score"]
)

print(
    "Clarity Score:",
    agent_result["clarity_score"]
)

print(
    "Coaching Tip:",
    agent_result["coaching_tip"]
)


Gemini server is temporarily busy. Retrying... (1/3)

--- Customer Analysis ---
Sentiment: negative
Urgency: high
Escalation Risk: high
Key Issue: Internet service outage and long wait time

--- Suggested Reply ---
I am truly sorry for the delay and completely understand how frustrating it is to be without internet for hours. 

To help resolve this, could you please provide your account number or service address? I will check your connection status immediately and investigate what is causing the delay.

--- Agent Coaching ---
Tone Score: 4
Empathy Score: 1
Clarity Score: 8
Coaching Tip: Acknowledge the customer's frustration and apologize for the long wait time before giving troubleshooting instructions.


In [20]:
# 5. DEMO MODE

def run_demo():

    print("\n")
    print("=" * 60)
    print("AI CUSTOMER SUPPORT COACH")
    print("=" * 60)

    session = RealTimeCoachingSession()

    # ==========================================
    # CUSTOMER MESSAGE 1
    # ==========================================

    customer_result = session.on_customer_message(
        "I have been waiting for my refund for 20 days! "
        "This is ridiculous. I want my money back immediately!"
    )

    print("\n--- Customer Analysis ---")

    print(
        "Sentiment:",
        customer_result["sentiment"]
    )

    print(
        "Urgency:",
        customer_result["urgency"]
    )

    print(
        "Escalation Risk:",
        customer_result["escalation_risk"]
    )

    print(
        "Key Issue:",
        customer_result["key_issue"]
    )

    if customer_result["suggested_reply"]:

        print("\n--- Suggested Reply ---")

        print(
            customer_result["suggested_reply"]
        )

    # ==========================================
    # AGENT RESPONSE 1
    # ==========================================

    agent_result = session.on_agent_message(
        "Please wait. We are checking your refund."
    )

    print("\n--- Agent Coaching ---")

    print(
        "Tone Score:",
        agent_result["tone_score"]
    )

    print(
        "Empathy Score:",
        agent_result["empathy_score"]
    )

    print(
        "Clarity Score:",
        agent_result["clarity_score"]
    )

    print(
        "Coaching Tip:",
        agent_result["coaching_tip"]
    )

    # ==========================================
    # CUSTOMER MESSAGE 2
    # ==========================================

    customer_result = session.on_customer_message(
        "How much longer do I have to wait?"
    )

    print("\n--- Customer Analysis ---")

    print(
        "Sentiment:",
        customer_result["sentiment"]
    )

    print(
        "Urgency:",
        customer_result["urgency"]
    )

    print(
        "Escalation Risk:",
        customer_result["escalation_risk"]
    )

    print(
        "Key Issue:",
        customer_result["key_issue"]
    )

    if customer_result["suggested_reply"]:

        print("\n--- Suggested Reply ---")

        print(
            customer_result["suggested_reply"]
        )

    # ==========================================
    # AGENT RESPONSE 2
    # ==========================================

    agent_result = session.on_agent_message(
        "I completely understand your frustration. "
        "Waiting 20 days for a refund is longer than expected. "
        "I will check the refund status and help you with the "
        "next steps."
    )

    print("\n--- Agent Coaching ---")

    print(
        "Tone Score:",
        agent_result["tone_score"]
    )

    print(
        "Empathy Score:",
        agent_result["empathy_score"]
    )

    print(
        "Clarity Score:",
        agent_result["clarity_score"]
    )

    print(
        "Coaching Tip:",
        agent_result["coaching_tip"]
    )


# ==========================================
# INTERACTIVE MODE
# ==========================================

def run_interactive():

    print("\n")
    print("=" * 60)
    print("REAL-TIME CUSTOMER SUPPORT COACH")
    print("=" * 60)

    print("\nType:")
    print("customer: your message")
    print("agent: your response")
    print("quit: exit the program")

    session = RealTimeCoachingSession()

    while True:

        user_input = input("\n> ").strip()

        if user_input.lower() == "quit":

            print("Goodbye!")

            break

        if user_input.lower().startswith(
            "customer:"
        ):

            message = user_input[
                len("customer:")
            ].strip()

            if message:

                result = session.on_customer_message(
                    message
                )

                print(
                    "\n--- Customer Analysis ---"
                )

                print(
                    "Sentiment:",
                    result["sentiment"]
                )

                print(
                    "Urgency:",
                    result["urgency"]
                )

                print(
                    "Escalation Risk:",
                    result["escalation_risk"]
                )

                print(
                    "Key Issue:",
                    result["key_issue"]
                )

                if result["suggested_reply"]:

                    print(
                        "\n--- Suggested Reply ---"
                    )

                    print(
                        result["suggested_reply"]
                    )

        elif user_input.lower().startswith(
            "agent:"
        ):

            message = user_input[
                len("agent:")
            ].strip()

            if message:

                result = session.on_agent_message(
                    message
                )

                print(
                    "\n--- Agent Coaching ---"
                )

                print(
                    "Tone Score:",
                    result["tone_score"]
                )

                print(
                    "Empathy Score:",
                    result["empathy_score"]
                )

                print(
                    "Clarity Score:",
                    result["clarity_score"]
                )

                print(
                    "Coaching Tip:",
                    result["coaching_tip"]
                )

        else:

            print(
                "Please use 'customer:' or 'agent:'"
            )


# ==========================================
# PROGRAM ENTRY POINT
# ==========================================

if __name__ == "__main__":

    run_demo()

    # Uncomment the next line when you want
    # to use interactive mode.

    # run_interactive()



AI CUSTOMER SUPPORT COACH

--- Customer Analysis ---
Sentiment: negative
Urgency: high
Escalation Risk: high
Key Issue: Delayed refund after 20 days

--- Suggested Reply ---
I completely understand your frustration, and I am truly sorry for the long delay. Waiting 20 days for a refund is certainly not the experience we want you to have. 

I am looking into this right now. Could you please share your order or reference number? As soon as I have those details, I will check the exact status of your refund and provide you with an immediate update on how we can get this resolved for you.

--- Agent Coaching ---
Tone Score: 2
Empathy Score: 1
Clarity Score: 4
Coaching Tip: Acknowledge the 20-day delay and apologize for the frustration before asking the customer to hold, and provide a clear timeframe for when you will update them.

--- Customer Analysis ---
Sentiment: negative
Urgency: medium
Escalation Risk: medium
Key Issue: Excessive wait time and lack of status update

--- Agent Coachin